In [1]:
import os
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from transformers import CvtForImageClassification, AutoFeatureExtractor
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


c:\Users\ardac\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
DATA_DIR        = r"C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split"
OUTPUT_DIR      = r"C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21"
IMG_SIZE        = 224       # CvT-21 için standart boyut
BATCH_SIZE      = 8         # RTX 3050 4GB için güvenli değer
ACCUM_STEPS     = 4         # Efektif batch = 8x4 = 32
NUM_WORKERS     = 0         # Windows için 0
PIN_MEMORY      = True
USE_AMP         = True      # Mixed precision — VRAM tasarrufu
 
FREEZE_EPOCHS   = 5
FINETUNE_EPOCHS = 5
 
FREEZE_LR       = 1e-3
FINETUNE_LR     = 1e-5
 
AUGMENTATION_MODE = "both"  # "both" | "mixup" | "cutmix" | "none"
MIXUP_ALPHA     = 0.4
CUTMIX_ALPHA    = 1.0
MIXUP_PROB      = 0.5
 
SEED            = 42
SUBSET_RATIO    = 0.4

In [3]:
# ─────────────────────────────────────────────
# SEED & DEVICE
# ─────────────────────────────────────────────
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
 
set_seed(SEED)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Kullanılan cihaz: {device}")
if device.type == "cuda":
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
 
os.makedirs(OUTPUT_DIR, exist_ok=True)

✅ Kullanılan cihaz: cuda
   GPU : NVIDIA GeForce RTX 3050 Laptop GPU
   VRAM: 4.3 GB


In [5]:
# ─────────────────────────────────────────────
# TRANSFORM
# CvT-21 ImageNet normalizasyonu kullanır
# ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])
 
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

In [6]:
# ─────────────────────────────────────────────
# SAFE DATASET
# ─────────────────────────────────────────────
from torchvision.datasets import ImageFolder
 
class SafeImageFolder(ImageFolder):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        print(f"   🔍 Dosyalar doğrulanıyor: {self.root}")
        valid_samples = []
        skipped = 0
        for path, label in tqdm(self.samples, desc="   Taranıyor", leave=False):
            try:
                with open(path, 'rb') as f:
                    f.read(10)
                valid_samples.append((path, label))
            except (FileNotFoundError, OSError):
                skipped += 1
        if skipped > 0:
            print(f"   ⚠️  {skipped} dosya atlandı")
        self.samples = valid_samples
        self.targets = [s[1] for s in valid_samples]
        print(f"   ✅ Geçerli dosya sayısı: {len(self.samples)}")

In [7]:
# ─────────────────────────────────────────────
# DATASET & DATALOADER
# ─────────────────────────────────────────────
print("\n📂 Veri seti yükleniyor...")
train_dataset = SafeImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_dataset   = SafeImageFolder(os.path.join(DATA_DIR, "val"),   transform=val_test_transform)
test_dataset  = SafeImageFolder(os.path.join(DATA_DIR, "test"),  transform=val_test_transform)
 
NUM_CLASSES = len(train_dataset.classes)
print(f"   Sınıf sayısı : {NUM_CLASSES}")
print(f"   Train        : {len(train_dataset)} görsel")
print(f"   Validation   : {len(val_dataset)} görsel")
print(f"   Test         : {len(test_dataset)} görsel")
 
def get_weighted_sampler(dataset):
    targets = [s[1] for s in dataset.samples]
    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    sample_weights = [class_weights[t] for t in targets]
    return WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )
 
sampler = get_weighted_sampler(train_dataset)
subset_size = int(len(train_dataset) * SUBSET_RATIO)
subset_sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor([sampler.weights[i] for i in range(len(train_dataset))]),
    num_samples=subset_size,
    replacement=True
)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=subset_sampler,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
 
print(f"   Her epoch: {subset_size} görsel / {subset_size//BATCH_SIZE} batch")


📂 Veri seti yükleniyor...
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\train


   ⚠️  1 dosya atlandı
   ✅ Geçerli dosya sayısı: 34019
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\val


   ✅ Geçerli dosya sayısı: 2043
   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\test


   ✅ Geçerli dosya sayısı: 1979
   Sınıf sayısı : 47
   Train        : 34019 görsel
   Validation   : 2043 görsel
   Test         : 1979 görsel
   Her epoch: 13607 görsel / 1700 batch


In [8]:
# ─────────────────────────────────────────────
# MixUp / CutMix
# ─────────────────────────────────────────────
def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    idx = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[idx]
    return mixed_x, y, y[idx], lam
 
def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0)).to(x.device)
    _, _, H, W = x.size()
    cut_rat = np.sqrt(1 - lam)
    cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    mixed_x = x.clone()
    mixed_x[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed_x, y, y[idx], lam
 
def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [9]:
# ─────────────────────────────────────────────
# MODEL — CvT-21 (HuggingFace transformers)
#
# CvT-21 mimarisi:
#   model.cvt          → backbone (3 stage: stage 0/1/2)
#     .stage_0 / .stage_1 / .stage_2  → convolutional transformer blokları
#   model.layernorm    → son normalizasyon
#   model.classifier   → Linear(384, num_labels)  ← değiştirdiğimiz yer
# ─────────────────────────────────────────────
from transformers import CvtForImageClassification
import huggingface_hub
print("\n🔧 CvT-21 modeli yükleniyor (HuggingFace)...")

model = CvtForImageClassification.from_pretrained(
    'microsoft/cvt-21',
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True,
    use_safetensors=True,
    torch_dtype=torch.float32   # ← bunu ekle
)
model = model.to(device)


🔧 CvT-21 modeli yükleniyor (HuggingFace)...


Some weights of CvtForImageClassification were not initialized from the model checkpoint at microsoft/cvt-21 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([47]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 384]) in the checkpoint and torch.Size([47, 384]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# ─────────────────────────────────────────────
# FREEZE / UNFREEZE
#
# CvT-21 katman yapısı:
#   model.cvt.stage_0  → ilk stage (en düşük seviye özellikler)
#   model.cvt.stage_1  → orta stage
#   model.cvt.stage_2  → son stage (en yüksek seviye özellikler)
#   model.layernorm    → normalizasyon
#   model.classifier   → sınıflandırıcı başlığı
# ─────────────────────────────────────────────
def freeze_backbone(model):
    """Sadece classifier'ı eğitilebilir bırak, cvt backbone'u dondur"""
    for param in model.cvt.parameters():
        param.requires_grad = False
    for param in model.layernorm.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True
 
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"   ❄️  Backbone donduruldu.")
    print(f"       Eğitilebilir: {trainable:,} / Toplam: {total:,} parametre")
 
def unfreeze_all(model):
    """Tüm parametreleri aç"""
    for param in model.parameters():
        param.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"   🔥 Tüm model açıldı. Eğitilebilir parametre: {trainable:,}")

In [11]:
# ─────────────────────────────────────────────
# EVALUATE
# ─────────────────────────────────────────────
def evaluate(model, loader, criterion, phase="val"):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    torch.cuda.empty_cache()
 
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                # HuggingFace modeli pixel_values ile çalışır
                outputs = model(pixel_values=images)
                logits  = outputs.logits
                loss    = criterion(logits, labels)
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            del images, labels, outputs, logits, loss
 
    torch.cuda.empty_cache()
    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
 
    return avg_loss, acc, prec, rec, f1, all_preds, all_labels
 

In [12]:
# ─────────────────────────────────────────────
# CONFUSION MATRIX KAYDET
# ─────────────────────────────────────────────
def save_confusion_matrix(labels, preds, class_names, save_path, title="Confusion Matrix"):
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(28, 24))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, annot_kws={"size": 6})
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Tahmin", fontsize=10)
    ax.set_ylabel("Gerçek", fontsize=10)
    plt.xticks(rotation=90, fontsize=6)
    plt.yticks(rotation=0, fontsize=6)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"   💾 Confusion matrix: {save_path}")

In [13]:
# ─────────────────────────────────────────────
# METRİK GRAFİĞİ
# ─────────────────────────────────────────────
def save_metrics_plot(history, save_path, title="Eğitim Grafiği"):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(title, fontsize=14)
    metrics = [
        ('train_loss', 'val_loss', 'Loss'),
        ('train_acc',  'val_acc',  'Accuracy'),
        ('train_f1',   'val_f1',   'F1 Score'),
        ('train_prec', 'val_prec', 'Precision'),
        ('train_rec',  'val_rec',  'Recall'),
    ]
    for i, (tr_key, val_key, label) in enumerate(metrics):
        ax = axes[i // 3][i % 3]
        ax.plot(epochs, history[tr_key], 'b-o', label='Train')
        ax.plot(epochs, history[val_key], 'r-o', label='Val')
        ax.set_title(label); ax.set_xlabel('Epoch')
        ax.legend(); ax.grid(True)
    axes[1][2].axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"   💾 Grafik: {save_path}")

In [14]:
# ─────────────────────────────────────────────
# EĞİTİM FONKSİYONU
# ─────────────────────────────────────────────
def train_phase(model, train_loader, val_loader, optimizer, scheduler,
                criterion, num_epochs, phase_name, aug_mode, class_names):
 
    history = {k: [] for k in ['train_loss','val_loss','train_acc','val_acc',
                                'train_f1','val_f1','train_prec','val_prec',
                                'train_rec','val_rec']}
    best_val_f1    = 0.0
    best_model_path = os.path.join(OUTPUT_DIR, f"best_{phase_name}.pth")
 
    cm_dir = os.path.join(OUTPUT_DIR, f"confusion_matrices_{phase_name}")
    os.makedirs(cm_dir, exist_ok=True)
 
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
 
    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        all_preds, all_labels = [], []
        start = time.time()
        torch.cuda.empty_cache()
 
        skipped_batches = 0
        optimizer.zero_grad()
        pbar = tqdm(train_loader, desc=f"[{phase_name}] Epoch {epoch}/{num_epochs}")
 
        for batch_idx, batch_data in enumerate(pbar):
            try:
                images, labels = batch_data
                images, labels = images.to(device), labels.to(device)
 
                use_aug = aug_mode != "none"
                y_a, y_b, lam = labels, labels, 1.0
                if use_aug:
                    use_mixup = (random.random() < MIXUP_PROB) if aug_mode == "both" else (aug_mode == "mixup")
                    if use_mixup:
                        images, y_a, y_b, lam = mixup_data(images, labels, MIXUP_ALPHA)
                    else:
                        images, y_a, y_b, lam = cutmix_data(images, labels, CUTMIX_ALPHA)
 
                with torch.autocast(device_type="cuda", enabled=USE_AMP):
                    # HuggingFace: pixel_values kwarg kullan
                    outputs = model(pixel_values=images)
                    logits  = outputs.logits
                    if use_aug and aug_mode != "none":
                        loss = mixed_criterion(criterion, logits, y_a, y_b, lam)
                    else:
                        loss = criterion(logits, labels)
                    loss = loss / ACCUM_STEPS
 
                scaler.scale(loss).backward()
 
                if (batch_idx + 1) % ACCUM_STEPS == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
 
                total_loss += loss.item() * ACCUM_STEPS
                preds = logits.argmax(dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                pbar.set_postfix({'loss': f'{loss.item() * ACCUM_STEPS:.4f}'})
                del images, labels, outputs, logits, loss
 
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"\n   ⚠️  OOM! Batch {batch_idx} atlanıyor...")
                    torch.cuda.empty_cache()
                    optimizer.zero_grad()
                    skipped_batches += 1
                    continue
                else:
                    raise e
            except Exception:
                skipped_batches += 1
                continue
 
        if skipped_batches > 0:
            print(f"   {skipped_batches} batch atlandı")
 
        if scheduler:
            scheduler.step()
 
        torch.cuda.empty_cache()
 
        tr_loss = total_loss / len(train_loader)
        tr_acc  = accuracy_score(all_labels, all_preds)
        tr_f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        tr_prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
        tr_rec  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
 
        val_loss, val_acc, val_prec, val_rec, val_f1, val_preds, val_labels = evaluate(
            model, val_loader, criterion
        )
 
        elapsed = time.time() - start
        print(f"\n📊 [{phase_name}] Epoch {epoch}/{num_epochs} ({elapsed:.0f}s)")
        print(f"   Train → Loss:{tr_loss:.4f}  Acc:{tr_acc:.4f}  F1:{tr_f1:.4f}  Prec:{tr_prec:.4f}  Rec:{tr_rec:.4f}")
        print(f"   Val   → Loss:{val_loss:.4f}  Acc:{val_acc:.4f}  F1:{val_f1:.4f}  Prec:{val_prec:.4f}  Rec:{val_rec:.4f}")
 
        history['train_loss'].append(tr_loss);  history['val_loss'].append(val_loss)
        history['train_acc'].append(tr_acc);    history['val_acc'].append(val_acc)
        history['train_f1'].append(tr_f1);      history['val_f1'].append(val_f1)
        history['train_prec'].append(tr_prec);  history['val_prec'].append(val_prec)
        history['train_rec'].append(tr_rec);    history['val_rec'].append(val_rec)
 
        # Train CM
        save_confusion_matrix(
            all_labels, all_preds, class_names,
            os.path.join(cm_dir, f"epoch{epoch:02d}_train_cm.png"),
            title=f"[{phase_name}] Train CM — Epoch {epoch}/{num_epochs} | F1:{tr_f1:.4f}"
        )
        # Val CM
        save_confusion_matrix(
            val_labels, val_preds, class_names,
            os.path.join(cm_dir, f"epoch{epoch:02d}_val_cm.png"),
            title=f"[{phase_name}] Val CM — Epoch {epoch}/{num_epochs} | F1:{val_f1:.4f}"
        )
 
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), best_model_path)
            print(f"   ✅ En iyi model kaydedildi (Val F1: {best_val_f1:.4f})")
 
    return history, best_model_path

In [15]:
# ─────────────────────────────────────────────
# TEST DEĞERLENDİRME
# ─────────────────────────────────────────────
def test_evaluation(model, test_loader, criterion, class_names, phase_label, aug_label):
    print(f"\n{'='*60}")
    print(f"🧪 TEST DEĞERLENDİRMESİ - {phase_label} | Aug: {aug_label}")
    print(f"{'='*60}")
    _, acc, prec, rec, f1, preds, labels = evaluate(model, test_loader, criterion, "test")
    print(f"   Accuracy  : {acc:.4f}")
    print(f"   Precision : {prec:.4f}")
    print(f"   Recall    : {rec:.4f}")
    print(f"   F1 Score  : {f1:.4f}")
 
    cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{phase_label}_{aug_label}.png")
    save_confusion_matrix(labels, preds, class_names, cm_path,
                          title=f"Test CM - {phase_label} | {aug_label}")
 
    report = classification_report(labels, preds, target_names=class_names, zero_division=0)
    report_path = os.path.join(OUTPUT_DIR, f"classification_report_{phase_label}_{aug_label}.txt")
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(f"Phase: {phase_label} | Augmentation: {aug_label}\n\n")
        f.write(report)
    print(f"   💾 Classification report: {report_path}")
 
    return acc, prec, rec, f1

In [16]:
# ─────────────────────────────────────────────
# GRAD-CAM
#
# CvT-21 Transformer tabanlı bir model olduğu için
# pytorch_grad_cam'ın standart GradCAM'i yerine
# GradCAMPlusPlus + reshape_transform kullanıyoruz.
#
# CvT son stage çıktısı (stage_2) shape:
#   (batch, H*W, embed_dim) → reshape ile (batch, embed_dim, H, W)
# ─────────────────────────────────────────────
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
 
GRADCAM_DIR      = os.path.join(OUTPUT_DIR, "gradcam")
SAMPLES_PER_CLASS = 2
os.makedirs(GRADCAM_DIR, exist_ok=True)
 
vis_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])
norm_transform = transforms.Normalize([0.485, 0.456, 0.406],
                                       [0.229, 0.224, 0.225])
 
def cvt_reshape_transform(tensor, height=7, width=7):
    """
    CvT son katman çıktısı: (batch, height*width, channels)
    GradCAM için: (batch, channels, height, width) gerekli
    """
    result = tensor.reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.transpose(2, 3).transpose(1, 2)
    return result
 
def get_target_layer_cvt(model):
    """
    CvT-21 için Grad-CAM hedef katmanı:
    model.cvt.encoder.stages[-1] → son transformer stage
    """
    return [model.cvt.encoder.stages[-1]]
 
def run_gradcam(model, model_name, dataset, class_names):
    model.eval()
    target_layers = get_target_layer_cvt(model)
    cam = GradCAMPlusPlus(
        model=model,
        target_layers=target_layers,
        reshape_transform=cvt_reshape_transform
    )
 
    save_dir = os.path.join(GRADCAM_DIR, model_name)
    os.makedirs(save_dir, exist_ok=True)
 
    class_samples = {i: [] for i in range(len(class_names))}
    for path, label in dataset.samples:
        if len(class_samples[label]) < SAMPLES_PER_CLASS:
            class_samples[label].append(path)
        if all(len(v) >= SAMPLES_PER_CLASS for v in class_samples.values()):
            break
 
    print(f"\n🎨 [{model_name}] Grad-CAM oluşturuluyor...")
    for class_idx, paths in tqdm(class_samples.items(), desc=model_name):
        class_name = class_names[class_idx]
        fig, axes = plt.subplots(len(paths), 3, figsize=(12, 4 * len(paths)))
        if len(paths) == 1:
            axes = [axes]
        fig.suptitle(f"{model_name} | Sınıf: {class_name}", fontsize=11)
 
        for row, path in enumerate(paths):
            try:
                from PIL import Image as PILImage
                pil_img = PILImage.open(path).convert("RGB")
                pil_img = pil_img.resize((IMG_SIZE, IMG_SIZE))
                rgb_img = np.array(pil_img).astype(np.float32) / 255.0
 
                input_tensor = norm_transform(vis_transform(pil_img)).unsqueeze(0).to(device)
 
                with torch.no_grad():
                    output = model(pixel_values=input_tensor)
                pred_idx   = output.logits.argmax(dim=1).item()
                pred_name  = class_names[pred_idx]
                confidence = torch.softmax(output.logits, dim=1)[0][pred_idx].item()
 
                targets = [ClassifierOutputTarget(class_idx)]
                grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
                grayscale_cam = grayscale_cam[0]
                cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
 
                is_correct   = (pred_idx == class_idx)
                border_color = "green" if is_correct else "red"
                status       = "✅ Doğru" if is_correct else f"❌ Yanlış → {pred_name}"
 
                axes[row][0].imshow(pil_img)
                axes[row][0].set_title("Orijinal", fontsize=9)
                axes[row][0].axis("off")
 
                axes[row][1].imshow(grayscale_cam, cmap="jet")
                axes[row][1].set_title("Grad-CAM++ Isı Haritası", fontsize=9)
                axes[row][1].axis("off")
 
                axes[row][2].imshow(cam_image)
                axes[row][2].set_title(f"{status}\nGüven: {confidence:.2%}",
                                       fontsize=9, color=border_color)
                axes[row][2].axis("off")
                for spine in axes[row][2].spines.values():
                    spine.set_edgecolor(border_color)
                    spine.set_linewidth(3)
 
            except Exception as e:
                print(f"   ⚠️  {path} atlandı: {e}")
                continue
 
        plt.tight_layout()
        safe_name = class_name.replace("/", "_").replace("\\", "_")
        plt.savefig(os.path.join(save_dir, f"{safe_name}.png"), dpi=100, bbox_inches="tight")
        plt.close()
 
    print(f"   💾 Kaydedildi: {save_dir}")

In [17]:
# ─────────────────────────────────────────────
# ANA AKIŞ
# ─────────────────────────────────────────────
criterion   = nn.CrossEntropyLoss()
class_names = train_dataset.classes
 
print(f"\n{'='*60}")
print(f"🚀 EĞİTİM BAŞLIYOR")
print(f"   Model          : CvT-21 (microsoft/cvt-21)")
print(f"   Görsel boyutu  : {IMG_SIZE}x{IMG_SIZE}")
print(f"   Augmentation   : {AUGMENTATION_MODE}")
print(f"   Freeze Epochs  : {FREEZE_EPOCHS}")
print(f"   Finetune Epochs: {FINETUNE_EPOCHS}")
print(f"   Batch Size     : {BATCH_SIZE}  (efektif: {BATCH_SIZE*ACCUM_STEPS})")
print(f"   Mixed Precision: {USE_AMP}")
print(f"{'='*60}\n")


🚀 EĞİTİM BAŞLIYOR
   Model          : CvT-21 (microsoft/cvt-21)
   Görsel boyutu  : 224x224
   Augmentation   : both
   Freeze Epochs  : 5
   Finetune Epochs: 5
   Batch Size     : 8  (efektif: 32)
   Mixed Precision: True



In [21]:
import gc
from PIL import Image as PILImage
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Hook'ları temizle
for stage in model.cvt.encoder.stages:
    for layer in stage.modules():
        layer._forward_hooks.clear()
        layer._forward_pre_hooks.clear()
gc.collect()
torch.cuda.empty_cache()

# Tüm parametreleri aç (gradyan akışı için şart)
for param in model.parameters():
    param.requires_grad = True

GRADCAM_DIR       = os.path.join(OUTPUT_DIR, "gradcam")
SAMPLES_PER_CLASS = 2
os.makedirs(GRADCAM_DIR, exist_ok=True)

vis_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])
norm_transform = transforms.Normalize(
    [0.485, 0.456, 0.406],
    [0.229, 0.224, 0.225]
)

class CvTWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        return self.model(pixel_values=x).logits

def cvt_reshape_transform(tensor, height=14, width=14):
    # (B, 197, 384) → cls token at → (B, 196, 384) → (B, 384, 14, 14)
    tensor = tensor[:, 1:, :]
    result = tensor.reshape(tensor.size(0), height, width, tensor.size(2))
    result = result.permute(0, 3, 1, 2)
    return result

def run_gradcam(model, model_name, dataset, class_names):
    model.eval()
    for param in model.parameters():
        param.requires_grad = True

    wrapped_model = CvTWrapper(model)
    # ✅ Çalışan katman: layers[14].layernorm_after
    target_layer  = model.cvt.encoder.stages[-1].layers[14].layernorm_after

    cam = GradCAM(
        model=wrapped_model,
        target_layers=[target_layer],
        reshape_transform=cvt_reshape_transform
    )

    save_dir = os.path.join(GRADCAM_DIR, model_name)
    os.makedirs(save_dir, exist_ok=True)

    # Her sınıftan örnek topla
    class_samples = {i: [] for i in range(len(class_names))}
    for path, label in dataset.samples:
        if len(class_samples[label]) < SAMPLES_PER_CLASS:
            class_samples[label].append(path)
        if all(len(v) >= SAMPLES_PER_CLASS for v in class_samples.values()):
            break

    print(f"\n🎨 [{model_name}] Grad-CAM oluşturuluyor...")
    for class_idx, paths in tqdm(class_samples.items(), desc=model_name):
        class_name = class_names[class_idx]
        fig, axes = plt.subplots(len(paths), 3, figsize=(12, 4 * len(paths)))
        if len(paths) == 1:
            axes = [axes]
        fig.suptitle(f"{model_name} | Sınıf: {class_name}", fontsize=11)

        for row, path in enumerate(paths):
            try:
                pil_img = PILImage.open(path).convert("RGB")
                pil_img = pil_img.resize((IMG_SIZE, IMG_SIZE))
                rgb_img = np.array(pil_img).astype(np.float32) / 255.0

                input_tensor = norm_transform(
                    vis_transform(pil_img)
                ).unsqueeze(0).to(device)

                with torch.no_grad():
                    output = model(pixel_values=input_tensor)
                pred_idx   = output.logits.argmax(dim=1).item()
                pred_name  = class_names[pred_idx]
                confidence = torch.softmax(output.logits, dim=1)[0][pred_idx].item()

                grayscale_cam = cam(
                    input_tensor=input_tensor,
                    targets=[ClassifierOutputTarget(class_idx)],
                    aug_smooth=False,
                    eigen_smooth=False
                )
                grayscale_cam = grayscale_cam[0]
                cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

                is_correct   = (pred_idx == class_idx)
                border_color = "green" if is_correct else "red"
                status       = "✅ Doğru" if is_correct else f"❌ Yanlış → {pred_name}"

                axes[row][0].imshow(pil_img)
                axes[row][0].set_title("Orijinal", fontsize=9)
                axes[row][0].axis("off")

                axes[row][1].imshow(grayscale_cam, cmap="jet")
                axes[row][1].set_title("Grad-CAM Isı Haritası", fontsize=9)
                axes[row][1].axis("off")

                axes[row][2].imshow(cam_image)
                axes[row][2].set_title(
                    f"{status}\nGüven: {confidence:.2%}",
                    fontsize=9, color=border_color
                )
                axes[row][2].axis("off")
                for spine in axes[row][2].spines.values():
                    spine.set_edgecolor(border_color)
                    spine.set_linewidth(3)

            except Exception as e:
                print(f"   ⚠️  {path} atlandı: {e}")
                continue

        plt.tight_layout()
        safe_name = class_name.replace("/", "_").replace("\\", "_")
        plt.savefig(
            os.path.join(save_dir, f"{safe_name}.png"),
            dpi=100, bbox_inches="tight"
        )
        plt.close()

    print(f"   💾 Kaydedildi: {save_dir}")

# ─── ÇALIŞTIR ───
best_freeze_path   = os.path.join(OUTPUT_DIR, "best_freeze.pth")
best_finetune_path = os.path.join(OUTPUT_DIR, "best_finetune.pth")

gradcam_dataset = SafeImageFolder(
    os.path.join(DATA_DIR, "test"),
    transform=val_test_transform
)

print("\n⚡ Phase 1 modeli yükleniyor...")
model.load_state_dict(torch.load(best_freeze_path, map_location=device))
run_gradcam(model, "Phase1_BackboneFreeze", gradcam_dataset, train_dataset.classes)

torch.cuda.empty_cache()

print("\n🔥 Phase 2 modeli yükleniyor...")
model.load_state_dict(torch.load(best_finetune_path, map_location=device))
run_gradcam(model, "Phase2_FineTune", gradcam_dataset, train_dataset.classes)

print(f"\n✅ GRAD-CAM TAMAMLANDI! → {GRADCAM_DIR}")

   🔍 Dosyalar doğrulanıyor: C:\Users\ardac\OneDrive\Masaüstü\YL_project\data_split\test


   ✅ Geçerli dosya sayısı: 1979

⚡ Phase 1 modeli yükleniyor...

🎨 [Phase1_BackboneFreeze] Grad-CAM oluşturuluyor...


Phase1_BackboneFreeze: 100%|██████████| 47/47 [01:54<00:00,  2.43s/it]


   💾 Kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\gradcam\Phase1_BackboneFreeze

🔥 Phase 2 modeli yükleniyor...

🎨 [Phase2_FineTune] Grad-CAM oluşturuluyor...


Phase2_FineTune: 100%|██████████| 47/47 [01:56<00:00,  2.48s/it]

   💾 Kaydedildi: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\gradcam\Phase2_FineTune

✅ GRAD-CAM TAMAMLANDI! → C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\gradcam


In [29]:
# ──────────────────────────────
# AŞAMA 1: BACKBONE FREEZE
# ──────────────────────────────
print("\n" + "="*60)
print("⚡ AŞAMA 1: BACKBONE FREEZE")
print("="*60)
freeze_backbone(model)
 
optimizer_freeze = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FREEZE_LR, weight_decay=1e-4
)
scheduler_freeze = optim.lr_scheduler.CosineAnnealingLR(optimizer_freeze, T_max=FREEZE_EPOCHS)
 
history_freeze, best_freeze_path = train_phase(
    model, train_loader, val_loader,
    optimizer_freeze, scheduler_freeze,
    criterion, FREEZE_EPOCHS,
    phase_name="freeze",
    aug_mode=AUGMENTATION_MODE,
    class_names=class_names
)
 
save_metrics_plot(history_freeze,
                  os.path.join(OUTPUT_DIR, "grafik_freeze.png"),
                  title=f"Aşama 1: Backbone Freeze | Aug: {AUGMENTATION_MODE}")
 
print("\n📌 Aşama 1 tamamlandı → Test seti üzerinde değerlendiriliyor...")
model.load_state_dict(torch.load(best_freeze_path, map_location=device))
acc1, prec1, rec1, f1_1 = test_evaluation(
    model, test_loader, criterion, class_names,
    phase_label="Phase1_Freeze", aug_label=AUGMENTATION_MODE
)


⚡ AŞAMA 1: BACKBONE FREEZE
   ❄️  Backbone donduruldu.
       Eğitilebilir: 18,095 / Toplam: 31,255,791 parametre


[freeze] Epoch 1/5:   0%|          | 0/1701 [00:00<?, ?it/s]

[freeze] Epoch 1/5: 100%|██████████| 1701/1701 [16:21<00:00,  1.73it/s, loss=2.5246]



📊 [freeze] Epoch 1/5 (1111s)
   Train → Loss:2.8240  Acc:0.2369  F1:0.2308  Prec:0.2318  Rec:0.2377
   Val   → Loss:1.4256  Acc:0.6084  F1:0.5416  Prec:0.5580  Rec:0.5613
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_freeze\epoch01_train_cm.png
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_freeze\epoch01_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.5416)


[freeze] Epoch 2/5: 100%|██████████| 1701/1701 [15:24<00:00,  1.84it/s, loss=2.5138]  


...


📊 [freeze] Epoch 3/5 (172s)
   Train → Loss:2.4055  Acc:0.3243  F1:0.3202  Prec:0.3244  Rec:0.3235
   Val   → Loss:1.2178  Acc:0.6608  F1:0.6009  Prec:0.6146  Rec:0.6181
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_freeze\epoch03_train_cm.png
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_freeze\epoch03_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.6009)


[freeze] Epoch 4/5: 100%|██████████| 1701/1701 [02:38<00:00, 10.72it/s, loss=2.3906]



📊 [freeze] Epoch 4/5 (172s)
   Train → Loss:2.3781  Acc:0.3328  F1:0.3296  Prec:0.3352  Rec:0.3317
   Val   → Loss:1.1579  Acc:0.6809  F1:0.6227  Prec:0.6330  Rec:0.6372
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_freeze\epoch04_train_cm.png
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_freeze\epoch04_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.6227)


[freeze] Epoch 5/5: 100%|██████████| 1701/1701 [02:53<00:00,  9.78it/s, loss=1.1400]



📊 [freeze] Epoch 5/5 (193s)
   Train → Loss:2.3528  Acc:0.3275  F1:0.3262  Prec:0.3352  Rec:0.3273
   Val   → Loss:1.1580  Acc:0.6774  F1:0.6209  Prec:0.6344  Rec:0.6348
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_freeze\epoch05_train_cm.png
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_freeze\epoch05_val_cm.png
   💾 Grafik: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\grafik_freeze.png

📌 Aşama 1 tamamlandı → Test seti üzerinde değerlendiriliyor...

🧪 TEST DEĞERLENDİRMESİ - Phase1_Freeze | Aug: both
   Accuracy  : 0.6796
   Precision : 0.6319
   Recall    : 0.6397
   F1 Score  : 0.6253
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrix_Phase1_Freeze_both.png
   💾 Classification report: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\classification_report_Phase1_Freeze_both.txt


In [30]:
# ──────────────────────────────
# AŞAMA 2: FINE-TUNING
# ──────────────────────────────
print("\n" + "="*60)
print("🔥 AŞAMA 2: FINE-TUNING (Backbone açılıyor)")
print("="*60)
unfreeze_all(model)
 
optimizer_finetune = optim.AdamW(
    model.parameters(),
    lr=FINETUNE_LR, weight_decay=1e-4
)
scheduler_finetune = optim.lr_scheduler.CosineAnnealingLR(optimizer_finetune, T_max=FINETUNE_EPOCHS)
 
history_finetune, best_finetune_path = train_phase(
    model, train_loader, val_loader,
    optimizer_finetune, scheduler_finetune,
    criterion, FINETUNE_EPOCHS,
    phase_name="finetune",
    aug_mode=AUGMENTATION_MODE,
    class_names=class_names
)
 
save_metrics_plot(history_finetune,
                  os.path.join(OUTPUT_DIR, "grafik_finetune.png"),
                  title=f"Aşama 2: Fine-Tuning | Aug: {AUGMENTATION_MODE}")
 
print("\n📌 Aşama 2 tamamlandı → Test seti üzerinde değerlendiriliyor...")
model.load_state_dict(torch.load(best_finetune_path, map_location=device))
acc2, prec2, rec2, f1_2 = test_evaluation(
    model, test_loader, criterion, class_names,
    phase_label="Phase2_FineTune", aug_label=AUGMENTATION_MODE
)


🔥 AŞAMA 2: FINE-TUNING (Backbone açılıyor)
   🔥 Tüm model açıldı. Eğitilebilir parametre: 31,255,791


[finetune] Epoch 1/5: 100%|██████████| 1701/1701 [06:25<00:00,  4.42it/s, loss=3.1208]



📊 [finetune] Epoch 1/5 (401s)
   Train → Loss:2.2384  Acc:0.3641  F1:0.3615  Prec:0.3682  Rec:0.3629
   Val   → Loss:0.8706  Acc:0.7406  F1:0.6972  Prec:0.7023  Rec:0.7046
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_finetune\epoch01_train_cm.png
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_finetune\epoch01_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.6972)


[finetune] Epoch 2/5: 100%|██████████| 1701/1701 [05:54<00:00,  4.79it/s, loss=2.7919]



📊 [finetune] Epoch 2/5 (369s)
   Train → Loss:2.1090  Acc:0.3833  F1:0.3839  Prec:0.3895  Rec:0.3845
   Val   → Loss:0.7993  Acc:0.7685  F1:0.7284  Prec:0.7318  Rec:0.7360
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_finetune\epoch02_train_cm.png
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_finetune\epoch02_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.7284)


...


📊 [finetune] Epoch 3/5 (368s)
   Train → Loss:2.0349  Acc:0.4196  F1:0.4218  Prec:0.4319  Rec:0.4196
   Val   → Loss:0.7726  Acc:0.7753  F1:0.7365  Prec:0.7411  Rec:0.7442
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_finetune\epoch03_train_cm.png
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_finetune\epoch03_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.7365)


[finetune] Epoch 4/5: 100%|██████████| 1701/1701 [05:55<00:00,  4.78it/s, loss=3.0954]



📊 [finetune] Epoch 4/5 (370s)
   Train → Loss:2.0068  Acc:0.4201  F1:0.4249  Prec:0.4404  Rec:0.4201
   Val   → Loss:0.7511  Acc:0.7773  F1:0.7389  Prec:0.7410  Rec:0.7466
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_finetune\epoch04_train_cm.png
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_finetune\epoch04_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.7389)


[finetune] Epoch 5/5: 100%|██████████| 1701/1701 [05:58<00:00,  4.75it/s, loss=1.6278]



📊 [finetune] Epoch 5/5 (373s)
   Train → Loss:1.9732  Acc:0.4241  F1:0.4304  Prec:0.4471  Rec:0.4242
   Val   → Loss:0.7237  Acc:0.7802  F1:0.7417  Prec:0.7450  Rec:0.7486
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_finetune\epoch05_train_cm.png
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrices_finetune\epoch05_val_cm.png
   ✅ En iyi model kaydedildi (Val F1: 0.7417)
   💾 Grafik: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\grafik_finetune.png

📌 Aşama 2 tamamlandı → Test seti üzerinde değerlendiriliyor...

🧪 TEST DEĞERLENDİRMESİ - Phase2_FineTune | Aug: both
   Accuracy  : 0.7676
   Precision : 0.7327
   Recall    : 0.7318
   F1 Score  : 0.7268
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrix_Phase2_FineTune_both.png
   💾 Classification report: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\classification_re

In [36]:
# ──────────────────────────────
# KARŞILAŞTIRMA TABLOSU
# (kernel restart sonrası kullan)
# ──────────────────────────────

best_freeze_path   = os.path.join(OUTPUT_DIR, "best_freeze.pth")
best_finetune_path = os.path.join(OUTPUT_DIR, "best_finetune.pth")
criterion = nn.CrossEntropyLoss()

# Phase 1 skorları
print("⚡ Phase 1 modeli değerlendiriliyor...")
model.load_state_dict(torch.load(best_freeze_path, map_location=device))
acc1, prec1, rec1, f1_1 = test_evaluation(
    model, test_loader, criterion, class_names,
    phase_label="Phase1_Freeze", aug_label=AUGMENTATION_MODE
)

torch.cuda.empty_cache()

# Phase 2 skorları
print("🔥 Phase 2 modeli değerlendiriliyor...")
model.load_state_dict(torch.load(best_finetune_path, map_location=device))
acc2, prec2, rec2, f1_2 = test_evaluation(
    model, test_loader, criterion, class_names,
    phase_label="Phase2_FineTune", aug_label=AUGMENTATION_MODE
)

torch.cuda.empty_cache()

# Tablo
print("\n" + "="*60)
print("📈 SONUÇ KARŞILAŞTIRMASI")
print("="*60)
print(f"{'Aşama':<30} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}")
print("-"*60)
print(f"{'Phase 1 (Freeze)':<30} {acc1:>8.4f} {prec1:>8.4f} {rec1:>8.4f} {f1_1:>8.4f}")
print(f"{'Phase 2 (Fine-Tune)':<30} {acc2:>8.4f} {prec2:>8.4f} {rec2:>8.4f} {f1_2:>8.4f}")
print("="*60)

# Özet dosyası
summary_path = os.path.join(OUTPUT_DIR, f"summary_{AUGMENTATION_MODE}.txt")
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(f"CvT-21 Eğitim Özeti\n")
    f.write(f"Augmentation: {AUGMENTATION_MODE}\n")
    f.write(f"Görsel boyutu: {IMG_SIZE}x{IMG_SIZE}\n")
    f.write(f"{'='*60}\n")
    f.write(f"{'Aşama':<30} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8}\n")
    f.write(f"{'-'*60}\n")
    f.write(f"{'Phase 1 (Freeze)':<30} {acc1:>8.4f} {prec1:>8.4f} {rec1:>8.4f} {f1_1:>8.4f}\n")
    f.write(f"{'Phase 2 (Fine-Tune)':<30} {acc2:>8.4f} {prec2:>8.4f} {rec2:>8.4f} {f1_2:>8.4f}\n")

print(f"\n✅ Tüm sonuçlar kaydedildi: {OUTPUT_DIR}")
print(f"   Özet: {summary_path}")
print("\n🎉 EĞİTİM TAMAMLANDI!")

⚡ Phase 1 modeli değerlendiriliyor...

🧪 TEST DEĞERLENDİRMESİ - Phase1_Freeze | Aug: both
   Accuracy  : 0.6796
   Precision : 0.6319
   Recall    : 0.6397
   F1 Score  : 0.6253
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrix_Phase1_Freeze_both.png
   💾 Classification report: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\classification_report_Phase1_Freeze_both.txt
🔥 Phase 2 modeli değerlendiriliyor...

🧪 TEST DEĞERLENDİRMESİ - Phase2_FineTune | Aug: both
   Accuracy  : 0.7676
   Precision : 0.7327
   Recall    : 0.7318
   F1 Score  : 0.7268
   💾 Confusion matrix: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\confusion_matrix_Phase2_FineTune_both.png
   💾 Classification report: C:\Users\ardac\OneDrive\Masaüstü\YL_project\outputs_cvt21\classification_report_Phase2_FineTune_both.txt

📈 SONUÇ KARŞILAŞTIRMASI
Aşama                               Acc     Prec      Rec       F1
------------------------------------------